# SimpleSignRecog：動画の前処理

手話動画から、学習に使う手指ランドマークを作ります。処理順は次のとおりです。

1. MediaPipeで左右21点のXYZ座標を抽出する
2. 検出できなかった座標を時間方向に補間する
3. 手の位置・大きさ・向きを正規化する

全件処理の前に、動画1本で各段階の結果を確認します。前回の実験と同じく、保存するNPZは抽出直後の126次元データです。欠損補間と正規化は学習時に適用します。

In [ ]:
# Colabに必要なライブラリを入れます。
%pip install -q mediapipe opencv-python-headless pandas scipy tqdm

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# リポジトリを取得します。すでに取得済みなら再利用します。
from pathlib import Path
import os
import subprocess

GITHUB_OWNER = 'YOUR_GITHUB_ACCOUNT'  # 自分のGitHubアカウント名に変更
REPOSITORY_URL = f'https://github.com/{GITHUB_OWNER}/SimpleSignRecog.git'
REPOSITORY_DIR = Path('/content/SimpleSignRecog')

if not REPOSITORY_DIR.exists():
    subprocess.run(['git', 'clone', REPOSITORY_URL, str(REPOSITORY_DIR)], check=True)
os.chdir(REPOSITORY_DIR)
print('作業ディレクトリ:', Path.cwd())

## データの場所

Google Driveの `MyDrive/SignRecogData/subject_videos/` に、クラス番号 `1`～`20` のディレクトリを置く想定です。異なる場所に置いた場合は、次のセルの `INPUT_DIR` を変更してください。

In [ ]:
SUBJECT_ID = 'subject_03'  # 匿名化した識別子
INPUT_DIR = Path('/content/drive/MyDrive/SignRecogData/subject_videos')
OUTPUT_ROOT = Path('/content/drive/MyDrive/SimpleSignRecogData')
RAW_OUTPUT_DIR = OUTPUT_ROOT / 'data' / SUBJECT_ID
CORRECTED_OUTPUT_DIR = OUTPUT_ROOT / 'corrected' / SUBJECT_ID

assert INPUT_DIR.is_dir(), f'入力ディレクトリがありません: {INPUT_DIR}'

counts = {class_id: len(list((INPUT_DIR / str(class_id)).glob('*.mp4')))
          for class_id in range(1, 21)}
print('クラスごとの動画数:', counts)
print('合計:', sum(counts.values()))
assert all(counts.values()), '動画が0本のクラスがあります。'

## 1本の動画で前処理を確認

配列の形は `(フレーム数, 126)` です。0～62列が左手、63～125列が右手です。

In [ ]:
import numpy as np

from src.preprocessing.landmark_extraction import extract_landmarks_from_video
from src.preprocessing.missing_data import interpolate_missing_data
from src.preprocessing.normalization import canonical_normalize_landmarks

sample_video = sorted(INPUT_DIR.rglob('*.mp4'))[0]
landmarks, had_inference, num_frames = extract_landmarks_from_video(sample_video)
interpolated = interpolate_missing_data(landmarks)
normalized = canonical_normalize_landmarks(interpolated)

print('動画:', sample_video)
print('抽出結果:', landmarks.shape, landmarks.dtype)
print('左右ラベルの推測:', had_inference)
print('抽出直後のNaN:', np.isnan(landmarks).sum())
print('補間後のNaN:', np.isnan(interpolated).sum())
print('正規化後のNaN:', np.isnan(normalized).sum())

In [ ]:
# 右手の手首X座標を例に、補間前後を描画します。
import matplotlib.pyplot as plt

frame_numbers = np.arange(len(landmarks))
plt.figure(figsize=(12, 4))
plt.plot(frame_numbers, landmarks[:, 63], 'o', markersize=3, label='抽出直後')
plt.plot(frame_numbers, interpolated[:, 63], '-', label='線形補間後')
plt.xlabel('フレーム番号')
plt.ylabel('右手首のX座標')
plt.grid(True)
plt.legend()
plt.show()

## 全動画からNPZを作成

このセルは419本すべてを処理するため時間がかかります。途中でColabの接続が切れても成果を残せるよう、出力先をGoogle Driveにしています。既存ファイルと同じ名前のNPZは上書きされます。

In [ ]:
from src.process_videos import create_dataset

metadata = create_dataset(
    input_root_dir=INPUT_DIR,
    output_base_dir=RAW_OUTPUT_DIR,
    max_num_hands=2,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5,
)
display(metadata.head())
print('作成したサンプル数:', len(metadata))

## 前回の実験と同じ左右補正

`quality_flag` が `inferred` の動画について、最初に手を検出したフレームの画面位置から左右を確認します。撮影者視点で左手が画面左、右手が画面右にあることを仮定した処理です。

In [ ]:
subprocess.run(
    [
        'python', 'src/correct_inferred_data.py',
        '--input_base_dir', str(RAW_OUTPUT_DIR),
        '--output_base_dir', str(CORRECTED_OUTPUT_DIR),
    ],
    check=True,
)
print('補正済みデータ:', CORRECTED_OUTPUT_DIR)